In [7]:
import json
import pandas as pd

PATH = "../data/artifacts/batch_outputs/p-m-s-l2-v4/threats_l0-batch_6a1ff58eb888819088417f0b1e2cb03f-output.jsonl"

rows = []
with open(PATH) as f:
    for line in f:
        rec = json.loads(line)
        custom_id = rec["custom_id"]
        status = rec["response"]["status_code"]
        try:
            outputs = rec["response"]["body"]["output"]
            msg = next(o for o in outputs if o["type"] == "message")
            text = msg["content"][0]["text"]
            parsed = json.loads(text)
            results = parsed.get("results", [])
            stop_reason = parsed.get("stop_reason", None)
        except Exception as e:
            results = None
            stop_reason = None
        rows.append({"custom_id": custom_id, "status": status, "results": results, "stop_reason": stop_reason})

df = pd.DataFrame(rows)
print(f"{len(df):,} records loaded")
df.head(10)

858 records loaded


,custom_id,status,results,stop_reason
0,WOS:001596379700001,200,[Unclear],stop
1,WOS:001606449300003,200,"[Invasive & Other Problematic Species, Genes &...",stop
2,WOS:001627095300001,200,"[Pollution, Climate Change & Severe Weather]",continue
3,WOS:001640033700001,200,"[Climate Change & Severe Weather, Natural Syst...",continue
4,WOS:001648933500001,200,[Climate Change & Severe Weather],continue
5,WOS:001655973900001,200,[Agriculture & Aquaculture],continue
6,WOS:001656137700001,200,[Pollution],stop
7,WOS:001657320400001,200,"[Invasive & Other Problematic Species, Genes &...",continue
8,WOS:001658870000001,200,[Climate Change & Severe Weather],continue
9,WOS:001659596300001,200,"[Agriculture & Aquaculture, Climate Change & S...",continue


In [8]:
# Value counts for stop_reason
df["stop_reason"].value_counts()

stop_reason
continue    731
stop        127
Name: count, dtype: int64

In [9]:
# Distribution of number of labels per record
df["n_labels"] = df["results"].apply(lambda x: len(x) if isinstance(x, list) else None)
df["n_labels"].value_counts().sort_index()

n_labels
1    740
2     89
3     22
4      7
Name: count, dtype: int64

In [10]:
# Frequency of each threat label
from collections import Counter

counter = Counter(label for labels in df["results"].dropna() for label in labels)
pd.Series(counter).sort_values(ascending=False)

Pollution                                                 263
Climate Change & Severe Weather                           177
Agriculture & Aquaculture                                 152
Unclear                                                    95
Invasive & Other Problematic Species, Genes & Diseases     92
Natural System Modifications                               61
Biological Resource Use                                    58
Residential & Commercial Development                       41
Energy Production & Mining                                 19
Human Intrusions & Disturbance                             18
Other Options                                              18
Transportation & Service Corridors                         18
dtype: int64

In [11]:
95+18

113

In [6]:
# Inspect records with a specific label
label_filter = "Not Applicable"
mask = df["results"].apply(lambda x: isinstance(x, list) and label_filter in x)
df[mask].head(10)

,custom_id,status,results,stop_reason,n_labels
41,WOS:001399307300001,200,[Not Applicable],stop,1
55,WOS:001451386700001,200,[Not Applicable],stop,1
71,WOS:001502704100007,200,[Not Applicable],stop,1
80,WOS:001517435200001,200,[Not Applicable],stop,1
106,WOS:001586174900009,200,[Not Applicable],stop,1
108,WOS:001594644500004,200,[Not Applicable],stop,1
189,WOS:001326611400001,200,[Not Applicable],stop,1
195,WOS:001341790500001,200,[Not Applicable],stop,1
199,WOS:001348068600024,200,[Not Applicable],stop,1
214,WOS:000824991300001,200,[Not Applicable],stop,1
